# Fine-tune NLLB on Deep Past train set (Kaggle, GPU)

Run this notebook **on Kaggle** with **GPU** enabled to fine-tune NLLB on the competition train data (Akkadian transliteration → English).

**Setup:** Right panel → **Settings** → **Accelerator** → **GPU T4 x2** (or any GPU). Internet can stay ON for this notebook.

**Inputs:** You must add the **competition dataset** and optionally your **NLLB dataset**. See "How to add inputs" below.

**Output:** The fine-tuned model is saved to `/kaggle/working/finetuned_nllb/`. After the run, go to **Output** → download that folder, zip it, create a **new Kaggle Dataset** from the zip, then add that dataset to your **submission notebook** as input (and use it instead of the base NLLB).

---

**How to add inputs to this notebook:**
1. In the **right panel**, find the **Input** (or **Data**) section.
2. Click **+ Add input** (or **Add data**).
3. **Competition data:** Search for the competition name (e.g. "deep past" or "deep-past-initiative-machine-translation") or go to the competition page → **Code** → **New Notebook** (that notebook will already have the competition data attached). If you created this notebook from the competition, the competition dataset may already be there; if not, add it by searching and clicking **Add**.
4. **Your NLLB model (optional):** Search for your dataset name (e.g. "nllb-200-distilled-600m") or open **Your work** / **My Datasets** and add it. If you don't add it, the notebook will download the base NLLB from Hugging Face (needs Internet ON).
5. **Save** the notebook. The data will appear under `/kaggle/input/<dataset-slug>/`.

## 1. Config and paths

In [ ]:
import os
COMPETITION_SLUG = "deep-past-initiative-machine-translation"
INPUT_DIR = f"/kaggle/input/{COMPETITION_SLUG}"
OUTPUT_DIR = "/kaggle/working"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "finetuned_nllb")

SRC_LANG = "arb_Arab"
TGT_LANG = "eng_Latn"
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
MAX_SAMPLES = 2000   # use None for full train; reduce if you hit OOM or want a quick run
BATCH_SIZE = 4
EPOCHS = 2
LR = 5e-5

## 2. Load train data

In [ ]:
import pandas as pd
# Find train.csv under /kaggle/input (Kaggle may mount competition data at different paths)
train_path = None
if os.path.isfile(os.path.join(INPUT_DIR, "train.csv")):
    train_path = os.path.join(INPUT_DIR, "train.csv")
else:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.json" in files:
            dirs.clear()
            continue
        if "train.csv" in files:
            train_path = os.path.join(root, "train.csv")
            break
if train_path is None:
    raise FileNotFoundError("train.csv not found under /kaggle/input. Add the competition dataset as input (right panel → + Add input → search competition name).")
df = pd.read_csv(train_path, encoding="utf-8", on_bad_lines="warn")
df = df.rename(columns={"transliteration": "source", "translation": "target"})
if "source" not in df.columns:
    df["source"] = df.iloc[:, 1]
if "target" not in df.columns:
    df["target"] = df.iloc[:, 2] if df.shape[1] > 2 else df.iloc[:, 1]
df = df[["source", "target"]].dropna()
if MAX_SAMPLES:
    df = df.head(MAX_SAMPLES)
print(f"Train samples: {len(df)}")
print(df.head(2))

## 3. Find base NLLB (dataset or Hugging Face) and load

In [ ]:
MODEL_PATH = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if COMPETITION_SLUG in dirs:
            dirs.remove(COMPETITION_SLUG)
        if "config.json" in files:
            MODEL_PATH = root
            break
if MODEL_PATH is None:
    MODEL_PATH = "facebook/nllb-200-distilled-600M"
print("Base model:", MODEL_PATH)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer_json = os.path.join(MODEL_PATH, "tokenizer.json") if os.path.isdir(MODEL_PATH) else None
if tokenizer_json and os.path.isfile(tokenizer_json):
    from transformers import PreTrainedTokenizerFast
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_json)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# Ensure pad token is set (required for DataCollatorForSeq2Seq and when loading from tokenizer.json only)
if tokenizer.pad_token is None or getattr(tokenizer, "pad_token_id", None) is None or tokenizer.pad_token_id < 0:
    if getattr(tokenizer, "eos_token", None) is not None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
if model.get_input_embeddings().num_embeddings < len(tokenizer):
    model.resize_token_embeddings(len(tokenizer))
print("Model and tokenizer loaded.")

## 4. Prepare dataset for Seq2Seq

In [ ]:
from datasets import Dataset

def preprocess(examples):
    if hasattr(tokenizer, "src_lang"):
        tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        examples["source"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False,
    )
    try:
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples["target"], max_length=MAX_TARGET_LENGTH, truncation=True, padding=False)
    except AttributeError:
        labels = tokenizer(examples["target"], max_length=MAX_TARGET_LENGTH, truncation=True, padding=False)
    label_ids = [[(x if x != tokenizer.pad_token_id else -100) for x in seq] for seq in labels["input_ids"]]
    inputs["labels"] = label_ids
    for k in list(inputs.keys()):
        if k not in ("input_ids", "attention_mask", "labels"):
            del inputs[k]
    return inputs

ds = Dataset.from_pandas(df)
ds = ds.map(preprocess, batched=True, remove_columns=ds.column_names)
ds.set_format(type="torch")
print(ds)

## 5. Train with Seq2SeqTrainer

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)

args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "train_out"),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=data_collator,
)
trainer.train()

## 6. Save model for submission notebook

In [ ]:
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print("Saved to", MODEL_SAVE_PATH)
print("Next: Download this folder from Output → zip → create Kaggle Dataset → add to submission notebook as input.")